# MODULE - 1 : PDF TEXT EXTRACTION using PyMuPDF and Tesseract

### a) Install PDF Extraction and OCR Dependencies
This cell installs Tesseract OCR and the Python libraries used to read PDFs, run OCR, and work with images. These tools are required before the extraction module can process both selectable text and text embedded in scanned pages.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr
!pip install -q pymupdf pytesseract pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


### b) Extract PDF Text, Layout, and Page Metadata
This module extracts words from a PDF using native PDF text extraction and OCR when needed. It records word positions and formatting/layout details, reduces duplicate text, and saves structured outputs for later NLP and sensitivity classification. It does not classify or encrypt the content.

Outputs: `tokens.csv`, `pdf_layout.json`, `page_renders/(page images)`

In [ ]:
#!/usr/bin/env python3
"""
Standalone PDF extraction module for semantic-aware PDF protection.

Outputs:
    tokens.csv
    pdf_layout.json
    page_renders/page_0001.png, ... (visual QA references)

Features:
    - Native text extraction using PyMuPDF
    - OCR using Tesseract
    - Native/OCR duplicate suppression
    - Word-level bounding boxes and reading-order approximation
    - Font, size, bold, italic, color and source metadata
    - Native vector fill inspection and separate background estimates
    - Page dimensions, rotation, image placements and diagnostics

This module does NOT classify sensitivity, encrypt or decrypt content.

Requirements:
    pip install pymupdf pytesseract pillow
    Install the Tesseract OCR application separately.
"""

from __future__ import annotations

import csv
import hashlib
import json
import logging
import math
import re
import shutil
import statistics
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import fitz
import pytesseract
from PIL import Image
from pytesseract import Output

LOGGER = logging.getLogger("pdf_extraction")
SCHEMA_VERSION = "1.0"


# ---------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------

def sha256_file(path: Path) -> str:
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def safe_float(value: Any) -> Optional[float]:
    try:
        number = float(value)
        return number if math.isfinite(number) else None
    except (TypeError, ValueError):
        return None


def rect_list(rect: Any) -> list[float]:
    return [round(float(value), 3) for value in rect]


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().casefold()


def color_to_rgb(value: Optional[int]) -> Optional[list[int]]:
    if value is None:
        return None

    return [
        (int(value) >> 16) & 255,
        (int(value) >> 8) & 255,
        int(value) & 255,
    ]


def rgb_to_hex(rgb: Optional[list[int]]) -> Optional[str]:
    if rgb is None:
        return None

    return "#{:02X}{:02X}{:02X}".format(*rgb)


def intersection_over_union(
    a: list[float],
    b: list[float],
) -> float:
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b

    ix0 = max(ax0, bx0)
    iy0 = max(ay0, by0)
    ix1 = min(ax1, bx1)
    iy1 = min(ay1, by1)

    intersection = max(0.0, ix1 - ix0) * max(
        0.0, iy1 - iy0
    )

    area_a = max(0.0, ax1 - ax0) * max(0.0, ay1 - ay0)
    area_b = max(0.0, bx1 - bx0) * max(0.0, by1 - by0)

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def approximately_same_box(
    a: list[float],
    b: list[float],
    tolerance: float = 5.0,
) -> bool:
    return all(
        abs(x - y) <= tolerance
        for x, y in zip(a, b)
    )


def check_tesseract() -> str:
    """
    Check that the Tesseract executable is installed and accessible.

    If needed on Windows, uncomment and modify:
        pytesseract.pytesseract.tesseract_cmd = (
            r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"
        )
    """
    if not shutil.which("tesseract"):
        # An explicitly configured pytesseract path may still work.
        configured = getattr(
            pytesseract.pytesseract,
            "tesseract_cmd",
            "tesseract",
        )
        if not Path(configured).is_file():
            raise RuntimeError(
                "Tesseract OCR was not found. Install Tesseract "
                "and add it to PATH, or configure tesseract_cmd "
                "inside this script."
            )

    try:
        return str(pytesseract.get_tesseract_version())
    except Exception as exc:
        raise RuntimeError(
            f"Tesseract could not start: {exc}"
        ) from exc


# ---------------------------------------------------------------------
# Native PDF text extraction
# ---------------------------------------------------------------------

def extract_native_words(
    page: fitz.Page,
    page_number: int,
) -> list[dict[str, Any]]:
    """
    Extract native words and associate them with their nearest
    overlapping PyMuPDF text span to obtain formatting metadata.

    PyMuPDF font flags:
        bit 1  (value 2): italic
        bit 4  (value 16): bold
    """
    page_dict = page.get_text("dict", sort=True)
    spans = []

    for block in page_dict.get("blocks", []):
        if block.get("type") != 0:
            continue

        for line in block.get("lines", []):
            direction = line.get("dir", [1.0, 0.0])

            for span in line.get("spans", []):
                color_value = span.get("color")
                color_rgb = color_to_rgb(color_value)

                spans.append({
                    "bbox": rect_list(
                        span.get("bbox", [0, 0, 0, 0])
                    ),
                    "font_family": span.get("font"),
                    "font_size": safe_float(span.get("size")),
                    "font_flags": span.get("flags"),
                    "text_color_rgb": color_rgb,
                    "text_color_hex": rgb_to_hex(color_rgb),
                    "line_direction": [
                        safe_float(direction[0]),
                        safe_float(direction[1]),
                    ],
                    "origin": span.get("origin"),
                    "text": span.get("text", ""),
                })

    result = []

    for order, item in enumerate(page.get_text("words", sort=True)):
        (
            x0, y0, x1, y1, text,
            block_no, line_no, word_no,
        ) = item[:8]

        bbox = [
            round(x0, 3),
            round(y0, 3),
            round(x1, 3),
            round(y1, 3),
        ]

        candidates = [
            span for span in spans
            if intersection_over_union(
                bbox, span["bbox"]
            ) > 0
        ]

        best_span = max(
            candidates,
            key=lambda span: intersection_over_union(
                bbox, span["bbox"]
            ),
            default=None,
        )

        flags = (
            best_span.get("font_flags")
            if best_span else None
        )

        result.append({
            "page": page_number,
            "word_index": order,
            "reading_order": None,
            "text": str(text),
            "bbox": bbox,
            "x0": bbox[0],
            "y0": bbox[1],
            "x1": bbox[2],
            "y1": bbox[3],
            "source": "native",
            "block_index": int(block_no),
            "line_index": int(line_no),
            "word_index_in_line": int(word_no),

            "font_family": (
                best_span.get("font_family")
                if best_span else None
            ),
            "font_size": (
                best_span.get("font_size")
                if best_span else None
            ),
            "font_flags": flags,
            "bold": (
                bool(int(flags) & 16)
                if flags is not None else None
            ),
            "italic": (
                bool(int(flags) & 2)
                if flags is not None else None
            ),
            "text_color_rgb": (
                best_span.get("text_color_rgb")
                if best_span else None
            ),
            "text_color_hex": (
                best_span.get("text_color_hex")
                if best_span else None
            ),
            "text_color_source": (
                "pdf_native_span_metadata"
                if best_span else None
            ),
            "style_metadata_source": (
                "pdf_native"
                if best_span else "unavailable"
            ),
            "ocr_confidence": None,
            "line_direction": (
                best_span.get("line_direction")
                if best_span else None
            ),
            "span_bbox": (
                best_span.get("bbox")
                if best_span else None
            ),
            "origin": (
                best_span.get("origin")
                if best_span else None
            ),
        })

    return result


# ---------------------------------------------------------------------
# Page rendering and OCR
# ---------------------------------------------------------------------

def render_page(
    page: fitz.Page,
    dpi: int,
) -> Image.Image:
    """Render a PDF page to an RGB image for OCR and visual QA."""
    scale = dpi / 72.0

    pixmap = page.get_pixmap(
        matrix=fitz.Matrix(scale, scale),
        colorspace=fitz.csRGB,
        alpha=False,
    )

    return Image.frombytes(
        "RGB",
        [pixmap.width, pixmap.height],
        pixmap.samples,
    )


def extract_ocr_words(
    image: Image.Image,
    page_number: int,
    dpi: int,
    language: str,
    psm: int,
    min_confidence: float,
) -> list[dict[str, Any]]:
    """
    OCR the entire rendered page, including image-baked text.

    OCR coordinates are converted from rendered pixels to PDF points.
    OCR does not recover the original font or exact PDF text color.
    """
    config = f"--oem 3 --psm {psm}"

    data = pytesseract.image_to_data(
        image,
        lang=language,
        config=config,
        output_type=Output.DICT,
    )

    scale = dpi / 72.0
    result = []

    for i, raw_text in enumerate(data["text"]):
        text = str(raw_text).strip()

        if not text:
            continue

        confidence = safe_float(data["conf"][i])

        if confidence is None or confidence < min_confidence:
            continue

        left = float(data["left"][i]) / scale
        top = float(data["top"][i]) / scale
        right = (
            float(data["left"][i]) + float(data["width"][i])
        ) / scale
        bottom = (
            float(data["top"][i]) + float(data["height"][i])
        ) / scale

        bbox = [
            round(left, 3),
            round(top, 3),
            round(right, 3),
            round(bottom, 3),
        ]

        result.append({
            "page": page_number,
            "word_index": None,
            "reading_order": None,
            "text": text,
            "bbox": bbox,
            "x0": bbox[0],
            "y0": bbox[1],
            "x1": bbox[2],
            "y1": bbox[3],
            "source": "ocr",
            "block_index": int(data["block_num"][i]),
            "line_index": int(data["line_num"][i]),
            "word_index_in_line": int(data["word_num"][i]),

            "font_family": None,
            "font_size": None,
            "font_flags": None,
            "bold": None,
            "italic": None,
            "text_color_rgb": None,
            "text_color_hex": None,
            "text_color_source": None,
            "style_metadata_source": "not_recoverable_from_ocr",
            "ocr_confidence": round(confidence, 2),
            "line_direction": None,
            "span_bbox": None,
            "origin": None,
        })

    return result


# ---------------------------------------------------------------------
# Native/OCR merging and duplicate suppression
# ---------------------------------------------------------------------

def merge_native_and_ocr(
    native_words: list[dict[str, Any]],
    ocr_words: list[dict[str, Any]],
    iou_threshold: float = 0.25,
) -> tuple[list[dict[str, Any]], dict[str, int]]:
    """
    Prefer native metadata when OCR detects the same word.

    OCR words are suppressed only when both normalized text and
    geometry appear to match. This reduces accidental deletion of
    repeated words at different locations.

    This is a heuristic; it cannot guarantee perfect deduplication.
    """
    merged = list(native_words)
    suppressed = 0

    # Build a normalized-text index to avoid comparing every OCR
    # word against every native word.
    native_by_text: dict[str, list[dict[str, Any]]] = {}

    for word in native_words:
        key = normalize_text(word["text"])
        native_by_text.setdefault(key, []).append(word)

    for ocr_word in ocr_words:
        key = normalize_text(ocr_word["text"])
        candidates = native_by_text.get(key, [])

        duplicate = False

        for native_word in candidates:
            overlap = intersection_over_union(
                native_word["bbox"],
                ocr_word["bbox"],
            )

            if (
                overlap >= iou_threshold
                or approximately_same_box(
                    native_word["bbox"],
                    ocr_word["bbox"],
                )
            ):
                duplicate = True
                break

        if duplicate:
            suppressed += 1
        else:
            merged.append(ocr_word)

    # Geometric ordering approximation. Multi-column and complex
    # documents may require more sophisticated line/column grouping.
    merged.sort(
        key=lambda word: (
            round(((word["y0"] + word["y1"]) / 2) / 4),
            word["x0"],
            word["y0"],
        )
    )

    for index, word in enumerate(merged):
        word["reading_order"] = index
        word["word_index"] = index

    counts = {
        "native_word_count": len(native_words),
        "ocr_word_count_before_dedup": len(ocr_words),
        "ocr_duplicates_suppressed": suppressed,
        "merged_word_count": len(merged),
    }

    return merged, counts


# ---------------------------------------------------------------------
# Background metadata
# ---------------------------------------------------------------------

def inspect_native_backgrounds(
    page: fitz.Page,
) -> dict[str, Any]:
    """
    Report inspectable PDF vector drawing fills.

    These are native PDF drawing properties, not a complete
    representation of every page background.
    """
    fills = []

    try:
        for drawing in page.get_drawings():
            fill = drawing.get("fill")
            rect = drawing.get("rect")

            if fill is None or rect is None:
                continue

            fills.append({
                "bbox": rect_list(rect),
                "fill_rgb_float": [
                    round(float(channel), 4)
                    for channel in fill
                ],
                "fill_opacity": safe_float(
                    drawing.get("fill_opacity")
                ),
                "drawing_type": drawing.get("type"),
                "source": "pdf_native_vector_drawing",
            })

    except Exception as exc:
        LOGGER.warning(
            "Could not inspect vector fills on page %s: %s",
            page.number + 1,
            exc,
        )

    return {
        "vector_fill_objects": fills,
        "note": (
            "Only inspectable vector fills are reported. Raster "
            "backgrounds, transparency, patterns, gradients, and "
            "complex clipping may not be fully represented."
        ),
    }


def sample_background(
    image: Image.Image,
    bbox: list[float],
    dpi: int,
) -> dict[str, Any]:
    """
    Estimate the background near a word by sampling surrounding
    rendered pixels. This is an estimate, not native PDF metadata.
    """
    scale = dpi / 72.0
    x0, y0, x1, y1 = bbox
    pad = max(2.0, 144.0 / dpi)

    # Sample strips outside the word rectangle, not the text itself.
    regions = [
        (x0 - pad, y0 - pad, x1 + pad, y0 - 0.3),
        (x0 - pad, y1 + 0.3, x1 + pad, y1 + pad),
        (x0 - pad, y0, x0 - 0.3, y1),
        (x1 + 0.3, y0, x1 + pad, y1),
    ]

    pixels = []

    for rx0, ry0, rx1, ry1 in regions:
        px0 = max(0, int(rx0 * scale))
        py0 = max(0, int(ry0 * scale))
        px1 = min(image.width, int(math.ceil(rx1 * scale)))
        py1 = min(image.height, int(math.ceil(ry1 * scale)))

        if px1 <= px0 or py1 <= py0:
            continue

        crop = image.crop((px0, py0, px1, py1))
        crop.thumbnail((80, 20))
        pixels.extend(crop.getdata())

    if not pixels:
        return {
            "rgb": None,
            "hex": None,
            "source": "pixel_sample_estimate",
            "confidence": "unavailable",
        }

    median_rgb = [
        int(statistics.median(pixel[channel] for pixel in pixels))
        for channel in range(3)
    ]

    spread = sum(
        max(pixel[channel] for pixel in pixels)
        - min(pixel[channel] for pixel in pixels)
        for channel in range(3)
    ) / 3

    if spread < 25:
        confidence = "higher"
    elif spread < 70:
        confidence = "medium"
    else:
        confidence = "low"

    return {
        "rgb": median_rgb,
        "hex": rgb_to_hex(median_rgb),
        "source": "pixel_sample_estimate",
        "confidence": confidence,
        "sampled_pixel_count": len(pixels),
    }


# ---------------------------------------------------------------------
# Image object metadata
# ---------------------------------------------------------------------

def inspect_image_placements(
    page: fitz.Page,
) -> list[dict[str, Any]]:
    images = []

    try:
        for image_info in page.get_images(full=True):
            xref = image_info[0]

            for rect in page.get_image_rects(xref):
                images.append({
                    "xref": xref,
                    "bbox": rect_list(rect),
                    "source": "pdf_image_object",
                })

    except Exception as exc:
        LOGGER.warning(
            "Could not inspect images on page %s: %s",
            page.number + 1,
            exc,
        )

    return images


# ---------------------------------------------------------------------
# CSV output
# ---------------------------------------------------------------------

CSV_FIELDS = [
    "page",
    "word_index",
    "reading_order",
    "text",
    "x0", "y0", "x1", "y1",
    "bbox",
    "source",
    "font_family",
    "font_size",
    "bold",
    "italic",
    "font_flags",
    "text_color_rgb",
    "text_color_hex",
    "text_color_source",
    "background_rgb_estimate",
    "background_hex_estimate",
    "background_source",
    "background_confidence",
    "ocr_confidence",
    "block_index",
    "line_index",
    "word_index_in_line",
    "style_metadata_source",
]


def word_to_csv_row(
    word: dict[str, Any],
) -> dict[str, Any]:
    background = word.get("background_estimate") or {}

    return {
        "page": word.get("page"),
        "word_index": word.get("word_index"),
        "reading_order": word.get("reading_order"),
        "text": word.get("text"),
        "x0": word.get("x0"),
        "y0": word.get("y0"),
        "x1": word.get("x1"),
        "y1": word.get("y1"),
        "bbox": json.dumps(word.get("bbox")),
        "source": word.get("source"),
        "font_family": word.get("font_family"),
        "font_size": word.get("font_size"),
        "bold": word.get("bold"),
        "italic": word.get("italic"),
        "font_flags": word.get("font_flags"),
        "text_color_rgb": json.dumps(
            word.get("text_color_rgb")
        ),
        "text_color_hex": word.get("text_color_hex"),
        "text_color_source": word.get("text_color_source"),
        "background_rgb_estimate": json.dumps(
            background.get("rgb")
        ),
        "background_hex_estimate": background.get("hex"),
        "background_source": background.get("source"),
        "background_confidence": background.get("confidence"),
        "ocr_confidence": word.get("ocr_confidence"),
        "block_index": word.get("block_index"),
        "line_index": word.get("line_index"),
        "word_index_in_line": word.get("word_index_in_line"),
        "style_metadata_source": word.get(
            "style_metadata_source"
        ),
    }


# ---------------------------------------------------------------------
# Main extraction pipeline
# ---------------------------------------------------------------------

def extract_pdf(
    input_pdf: Path,
    output_dir: Path,
    dpi: int = 300,
    language: str = "eng",
    psm: int = 3,
    min_confidence: float = 35.0,
    background_sampling: bool = True,
    save_page_renders: bool = True,
) -> tuple[Path, Path]:

    input_pdf = input_pdf.resolve()
    output_dir = output_dir.resolve()

    if not input_pdf.is_file():
        raise FileNotFoundError(
            f"Input PDF not found: {input_pdf}"
        )

    if input_pdf.suffix.lower() != ".pdf":
        raise ValueError("Input file must have a .pdf extension.")

    if not 100 <= dpi <= 600:
        raise ValueError("DPI must be between 100 and 600.")

    if not 0 <= min_confidence <= 100:
        raise ValueError(
            "Minimum OCR confidence must be between 0 and 100."
        )

    if not 0 <= psm <= 13:
        raise ValueError("PSM must be between 0 and 13.")

    tesseract_version = check_tesseract()

    output_dir.mkdir(parents=True, exist_ok=True)

    renders_dir = output_dir / "page_renders"

    if save_page_renders:
        renders_dir.mkdir(parents=True, exist_ok=True)

    try:
        document = fitz.open(input_pdf)
    except Exception as exc:
        raise RuntimeError(
            f"Could not open the PDF: {exc}"
        ) from exc

    if document.is_encrypted:
        document.close()
        raise RuntimeError(
            "The PDF is password-protected. Provide an accessible "
            "copy before extraction."
        )

    if document.page_count == 0:
        document.close()
        raise RuntimeError("The PDF contains no pages.")

    all_words = []
    page_records = []
    warnings = []

    total_native = 0
    total_ocr = 0
    total_duplicates = 0

    try:
        for page_index in range(document.page_count):
            page = document.load_page(page_index)
            page_number = page_index + 1

            LOGGER.info(
                "Processing page %s/%s",
                page_number,
                document.page_count,
            )

            width = float(page.rect.width)
            height = float(page.rect.height)
            rotation = int(page.rotation)

            native_words = extract_native_words(
                page, page_number
            )

            image = render_page(page, dpi)

            if save_page_renders:
                render_path = (
                    renders_dir / f"page_{page_number:04d}.png"
                )
                image.save(render_path, format="PNG")
                render_reference = str(
                    Path("page_renders") / render_path.name
                )
            else:
                render_reference = None

            # Full-page OCR finds text embedded in raster images.
            ocr_words = extract_ocr_words(
                image=image,
                page_number=page_number,
                dpi=dpi,
                language=language,
                psm=psm,
                min_confidence=min_confidence,
            )

            merged, counts = merge_native_and_ocr(
                native_words,
                ocr_words,
            )

            total_native += counts["native_word_count"]
            total_ocr += counts["ocr_word_count_before_dedup"]
            total_duplicates += counts[
                "ocr_duplicates_suppressed"
            ]

            for word in merged:
                if background_sampling:
                    word["background_estimate"] = sample_background(
                        image, word["bbox"], dpi
                    )
                else:
                    word["background_estimate"] = {
                        "rgb": None,
                        "hex": None,
                        "source": "not_sampled",
                        "confidence": "unavailable",
                    }

            all_words.extend(merged)

            native_characters = sum(
                len(word["text"]) for word in native_words
            )

            ocr_only_count = sum(
                word["source"] == "ocr" for word in merged
            )

            if not native_words and not ocr_words:
                warnings.append(
                    f"Page {page_number}: neither native extraction "
                    "nor OCR found any text."
                )

            if not native_words and ocr_only_count:
                warnings.append(
                    f"Page {page_number}: no native text layer was "
                    "detected; the page may be scanned."
                )

            if native_words and not ocr_words:
                warnings.append(
                    f"Page {page_number}: OCR found no words. "
                    "Inspect image quality and OCR settings."
                )

            if (
                merged
                and ocr_only_count / len(merged) > 0.65
            ):
                warnings.append(
                    f"Page {page_number}: high OCR-only ratio "
                    f"({ocr_only_count}/{len(merged)}). Review OCR."
                )

            if any(
                len(word["text"]) == 1
                and word["text"].isalnum()
                for word in merged
            ):
                warnings.append(
                    f"Page {page_number}: single-character tokens "
                    "exist; check for OCR/layout splitting."
                )

            page_records.append({
                "page": page_number,
                "width_points": round(width, 3),
                "height_points": round(height, 3),
                "rotation_degrees": rotation,
                "rotation_note": (
                    "Dimensions use PyMuPDF's displayed page "
                    "rectangle; rotation is also recorded separately."
                ),
                "mediabox": rect_list(page.mediabox),
                "cropbox": rect_list(page.cropbox),
                "page_render": render_reference,
                "render_dpi": dpi,
                "render_pixel_dimensions": list(image.size),
                "native_word_count": counts["native_word_count"],
                "ocr_word_count_before_dedup": counts[
                    "ocr_word_count_before_dedup"
                ],
                "ocr_duplicates_suppressed": counts[
                    "ocr_duplicates_suppressed"
                ],
                "merged_word_count": counts["merged_word_count"],
                "native_text_char_count": native_characters,
                "image_objects": inspect_image_placements(page),
                "background_metadata": inspect_native_backgrounds(
                    page
                ),
                "words": merged,
            })

    finally:
        document.close()

    # Write one CSV row per extracted word.
    csv_path = output_dir / "tokens.csv"

    with csv_path.open(
        "w",
        newline="",
        encoding="utf-8-sig",
    ) as stream:
        writer = csv.DictWriter(
            stream,
            fieldnames=CSV_FIELDS,
            extrasaction="ignore",
        )
        writer.writeheader()

        for word in all_words:
            writer.writerow(word_to_csv_row(word))

    source_counts = Counter(
        word["source"] for word in all_words
    )

    layout = {
        "schema_version": SCHEMA_VERSION,
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "input_pdf": {
            "filename": input_pdf.name,
            "sha256": sha256_file(input_pdf),
            "file_size_bytes": input_pdf.stat().st_size,
            "page_count": len(page_records),
            "preservation_warning": (
                "Keep the original PDF unchanged. The sidecar files "
                "are not a lossless reconstruction of the PDF."
            ),
        },

        "configuration": {
            "native_extractor": "PyMuPDF",
            "ocr_engine": "Tesseract via pytesseract",
            "tesseract_version": tesseract_version,
            "ocr_language": language,
            "ocr_page_segmentation_mode": psm,
            "render_dpi": dpi,
            "minimum_ocr_confidence": min_confidence,
            "background_sampling_enabled": background_sampling,
            "page_renders_saved": save_page_renders,
        },

        "coverage": {
            "total_words": len(all_words),
            "native_words": source_counts.get("native", 0),
            "ocr_words_kept_after_dedup": source_counts.get(
                "ocr", 0
            ),
            "native_words_before_merge": total_native,
            "ocr_words_before_dedup": total_ocr,
            "ocr_duplicates_suppressed": total_duplicates,
            "pages_with_words": sum(
                page["merged_word_count"] > 0
                for page in page_records
            ),
            "pages_without_words": sum(
                page["merged_word_count"] == 0
                for page in page_records
            ),
            "notes": [
                "Coverage counts do not prove that every visible "
                "word was captured.",
                "OCR confidence is a Tesseract estimate, not a "
                "calibrated probability.",
                "Reading order is a geometric approximation.",
                "Image-baked text is not native PDF text.",
            ],
        },

        "warnings": warnings,

        "visual_fidelity": {
            "recommended_restore_strategy": [
                "Preserve the exact original PDF bytes and SHA-256.",
                "Use word bounding boxes to locate text, but edit a "
                "copy of the original PDF rather than reconstructing "
                "the whole document from extracted words.",
                "OCR does not recover the original font, glyphs, "
                "vector objects, or exact text color.",
                "Saved page renders are visual references, not a "
                "replacement for the original PDF.",
            ],
            "background_color_distinction": {
                "native_pdf_data": (
                    "background_metadata.vector_fill_objects "
                    "contains inspectable native vector fills."
                ),
                "pixel_estimates": (
                    "background_estimate is sampled from rendered "
                    "pixels and may be affected by text, images, "
                    "antialiasing, transparency, and nearby graphics."
                ),
            },
        },

        "pages": page_records,
    }

    json_path = output_dir / "pdf_layout.json"

    with json_path.open("w", encoding="utf-8") as stream:
        json.dump(
            layout,
            stream,
            ensure_ascii=False,
            indent=2,
        )

    LOGGER.info(
        "Finished: %s total words; %s native; %s OCR-only; "
        "%s OCR duplicates suppressed.",
        len(all_words),
        source_counts.get("native", 0),
        source_counts.get("ocr", 0),
        total_duplicates,
    )

    LOGGER.info("CSV output: %s", csv_path)
    LOGGER.info("Layout output: %s", json_path)

    for warning in warnings:
        LOGGER.warning(warning)

    return csv_path, json_path


# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

def validate_outputs(
    csv_path: Path,
    json_path: Path,
) -> None:
    """
    Structural smoke test only. It does not certify OCR accuracy.
    """
    with csv_path.open(
        "r",
        newline="",
        encoding="utf-8-sig",
    ) as stream:
        rows = list(csv.DictReader(stream))

    with json_path.open("r", encoding="utf-8") as stream:
        layout = json.load(stream)

    assert isinstance(layout.get("pages"), list)
    assert (
        layout["input_pdf"]["page_count"]
        == len(layout["pages"])
    )
    assert layout["coverage"]["total_words"] == len(rows)

    for row in rows:
        assert row["text"].strip(), "Blank word row found."
        assert row["source"] in {"native", "ocr"}

        for field in ("x0", "y0", "x1", "y1"):
            assert row[field] != "", (
                f"Missing coordinate {field}"
            )

    print(
        f"Structural validation passed: {len(rows)} words, "
        f"{len(layout['pages'])} pages."
    )
    print(
        "Note: visually verify the OCR results separately."
    )

ModuleNotFoundError: No module named 'fitz'

In [ ]:
#Running this code
from pathlib import Path

extract_pdf(
    Path("/content/Medical_Data_Document.pdf"),
    Path("data/processed")
)

# MODULE - 2 : NLP using spaCy

### a) Install spaCy and Its English Language Model
This cell installs spaCy and downloads the small English model used for linguistic processing and named-entity recognition. The model helps identify features such as people, organizations, and locations in extracted tokens.

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

### b) Enrich Extracted Tokens with NLP Features
This cell loads the extracted token CSV and uses spaCy plus normalization rules to add features such as token type, stop-word/punctuation flags, currency, URLs, email addresses, and named entities. The enriched data is saved separately as `tokens_nlp.csv`, preserving the original extraction output.

In [ ]:

import re
import unicodedata
from pathlib import Path

import pandas as pd
import spacy

"""
Features:
- The code uses spaCy's like_url, like_email, like_num, is_currency, is_stop, is_punct, and is_alpha attributes wherever applicable.
- The currency fallback recognizes common currency codes and names in addition to currency symbols.
- NER produces both is_entity and entity_label, which can contain labels such as PERSON, ORG, and GPE.
  NOTE: Not always accurate.
- The output is saved to tokens_nlp.csv. Original tokens.csv remains unchanged.
"""
# --------------------------------------------------
# 1. PATHS AND CONFIGURATION
# --------------------------------------------------

INPUT_CSV = Path("/content/data/processed/tokens.csv")
OUTPUT_CSV = Path("/content/data/processed/tokens_nlp.csv")

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_CSV}")

nlp = spacy.load("en_core_web_sm")

df = pd.read_csv(INPUT_CSV, keep_default_na=False)

possible_token_columns = [
    "token", "text", "word", "token_text", "original_token"
]
token_col = next(
    (c for c in possible_token_columns if c in df.columns),
    None
)

if token_col is None:
    raise ValueError(
        f"Could not find the token column. "
        f"Available columns: {df.columns.tolist()}"
    )

possible_page_columns = [
    "page", "page_num", "page_number", "page_index"
]
page_col = next(
    (c for c in possible_page_columns if c in df.columns),
    None
)


# --------------------------------------------------
# 2. NORMALIZE TOKENS FOR DATASET LOOKUPS
# --------------------------------------------------

def make_base_word(value):
    """
    Create a normalized lookup value.
    Original token text and layout metadata remain unchanged.
    """
    text = unicodedata.normalize("NFKC", str(value)).strip().lower()

    # Remove surrounding punctuation, not internal punctuation.
    text = re.sub(
        r"^[^\w]+|[^\w]+$",
        "",
        text,
        flags=re.UNICODE
    )

    return text


df["base_word"] = df[token_col].map(make_base_word)


# --------------------------------------------------
# 3. CURRENCY FALLBACK
# --------------------------------------------------
# spaCy's token.is_currency detects currency symbols.
# This fallback additionally recognizes common currency
# codes and names when they appear as standalone tokens.

CURRENCY_CODES = {
    "USD", "EUR", "GBP", "INR", "JPY", "CNY", "CAD",
    "AUD", "CHF", "SGD", "NZD", "AED", "SAR", "BDT", "PKR"
}

CURRENCY_NAMES = {
    "dollar", "dollars", "euro", "euros",
    "pound", "pounds", "rupee", "rupees",
    "yen", "yuan", "franc", "francs",
    "dirham", "dirhams", "penny", "pence", "paise"
}


# --------------------------------------------------
# 4. SPACY TOKEN FEATURES
# --------------------------------------------------

def extract_token_features(original, base_word):
    original = str(original).strip()

    # Tokenize the original text using spaCy.
    doc = nlp.make_doc(original)
    tokens = [t for t in doc if not t.is_space]

    # Tokenize the normalized lookup value.
    base_doc = nlp.make_doc(base_word)
    base_tokens = [t for t in base_doc if not t.is_space]

    # Punctuation is true only when the original token
    # consists entirely of punctuation.
    is_punct = bool(tokens) and all(
        t.is_punct for t in tokens
    )

    # Use built-in lexical attributes.
    is_stopword = (
        len(base_tokens) == 1
        and base_tokens[0].is_stop
    )

    is_word = (
        len(base_tokens) == 1
        and base_tokens[0].is_alpha
    )

    is_num = any(t.like_num for t in tokens)

    is_url = any(t.like_url for t in tokens)

    is_email = any(t.like_email for t in tokens)

    # Built-in currency-symbol detection.
    is_currency = any(t.is_currency for t in tokens)

    # Fallback for standalone currency codes and names.
    normalized_original = original.strip(".,;:!?()[]{}\"'")
    if normalized_original.upper() in CURRENCY_CODES:
        is_currency = True

    if normalized_original.casefold() in CURRENCY_NAMES:
        is_currency = True

    return {
        "is_stopword": bool(is_stopword),
        "is_punct": bool(is_punct),
        "is_url": bool(is_url),
        "is_email": bool(is_email),
        "is_currency": bool(is_currency),
        "is_word": bool(is_word),
        "is_num": bool(is_num),
    }


feature_rows = [
    extract_token_features(original, base_word)
    for original, base_word in zip(df[token_col], df["base_word"])
]

features_df = pd.DataFrame(feature_rows, index=df.index)
df[features_df.columns] = features_df


# --------------------------------------------------
# 5. NAMED ENTITY RECOGNITION (NER)
# --------------------------------------------------

df["is_entity"] = False
df["entity_label"] = ""

if page_col is not None:
    page_groups = df.groupby(page_col, sort=False).groups.values()
else:
    page_groups = [df.index]

for group_indices in page_groups:
    indices = list(group_indices)
    texts = df.loc[indices, token_col].astype(str).tolist()

    # Reconstruct page text and preserve each row's character span.
    parts = []
    offsets = []
    cursor = 0

    for text in texts:
        if parts:
            parts.append(" ")
            cursor += 1

        start = cursor
        parts.append(text)
        cursor += len(text)
        end = cursor

        offsets.append((start, end))

    page_text = "".join(parts)

    if not page_text.strip():
        continue

    # Run the full NLP pipeline, including NER.
    doc = nlp(page_text)

    for ent in doc.ents:
        for row_index, (start, end) in zip(indices, offsets):
            if start < ent.end_char and end > ent.start_char:
                df.at[row_index, "is_entity"] = True

                existing = df.at[row_index, "entity_label"]
                labels = set(filter(None, existing.split("|")))
                labels.add(ent.label_)

                df.at[row_index, "entity_label"] = "|".join(
                    sorted(labels)
                )


# --------------------------------------------------
# 6. VALIDATE AND SAVE
# --------------------------------------------------

boolean_columns = [
    "is_stopword",
    "is_punct",
    "is_url",
    "is_email",
    "is_currency",
    "is_word",
    "is_num",
    "is_entity",
]

for col in boolean_columns:
    df[col] = df[col].astype(bool)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print("NLP feature extraction completed.")
print(f"Input tokens:  {len(df):,}")
print(f"Output file:   {OUTPUT_CSV}")
print(f"Token column:  {token_col}")
print(f"Page column:   {page_col or 'Not found; used one document'}")

print("\nFeature counts:")
for col in boolean_columns:
    print(f"{col:15s}: {int(df[col].sum()):,}")

print(f"\nDetected entity tokens: {int(df['is_entity'].sum()):,}")

NLP feature extraction completed.
Input tokens:  471
Output file:   /content/data/processed/tokens_nlp.csv
Token column:  text
Page column:   page

Feature counts:
is_stopword    : 54
is_punct       : 18
is_url         : 0
is_email       : 0
is_currency    : 0
is_word        : 337
is_num         : 75
is_entity      : 116

Detected entity tokens: 116


# MODULE - 3 : SENSITIVITY CLASSIFICATION using Deterministic Rules and Predefined Sensitivity Dataset



### a) Assign a Sensitivity Level to Each Token
This function combines the NLP-enriched tokens with the predefined sensitivity dataset and applies rule-based priorities. It assigns labels such as Low, Medium, or High and writes the result to `tokens_with_sensitivity.csv`, which the encryption stage uses to decide what content to protect.

In [ ]:
import os
import pandas as pd


def populate_token_sensitivity(
    input_dir="data/processed",
    combined_filename="combined_dataset.csv",
    tokens_filename="tokens_nlp.csv",
    output_filename="tokens_with_sensitivity.csv",
):
  # Construct full file paths
  combined_path = os.path.join(input_dir, combined_filename)
  tokens_path = os.path.join(input_dir, tokens_filename)
  output_path = os.path.join(input_dir, output_filename)

  # Load the datasets
  combined_df = pd.read_csv(combined_path)
  tokens_df = pd.read_csv(tokens_path)

  # Create a lookup dictionary from the combined dataset (token -> sensitivity_level)
  sensitivity_map = dict(
      zip(combined_df["token"], combined_df["sensitivity_level"])
  )

  def apply_rules(row):
    # 1. Apply predefined rules first
    is_stopword = bool(row.get("is_stopword", False))
    is_punct = bool(row.get("is_punct", False))
    is_num = bool(row.get("is_num", False))
    is_currency = bool(row.get("is_currency", False))
    is_url = bool(row.get("is_url", False))
    is_email = bool(row.get("is_email", False))
    is_entity = bool(row.get("is_entity", False))

    # High sensitivity rule flags
    if is_num or is_currency or is_url or is_email or is_entity:
      return "High"

    # Low sensitivity rule flags (both or either? The prompt specifies "is_stopword, is_punct" to low)
    if is_stopword or is_punct:
      return "Low"

    # 2. Check the base word field in the combined dataset
    base_word = row.get("base_word")

    if pd.notna(base_word) and base_word in sensitivity_map:
      return sensitivity_map[base_word]

    # 3. Default fallback if token/base word is not found in the combined dataset
    return "High"

  # Apply the rules and lookup to populate the sensitivity column
  tokens_df["sensitivity"] = tokens_df.apply(apply_rules, axis=1)

  # Ensure output directory exists and save the file
  os.makedirs(input_dir, exist_ok=True)
  tokens_df.to_csv(output_path, index=False)
  print(f"Processing complete. Updated file saved as '{output_path}'.")


if __name__ == "__main__":
  populate_token_sensitivity()

Processing complete. Updated file saved as 'data/processed/tokens_with_sensitivity.csv'.


# MODULE - 4 : CRYPTOGRAPHY

### a) Install Cryptography and PDF-Processing Dependencies
This cell installs the libraries needed for PDF handling, image processing, data loading, and cryptographic operations. They are used by the key-generation, encryption, and decryption stages.

In [ ]:
pip install pymupdf pandas pillow cryptography

### b) Generate and Save an RSA Public/Private Key Pair
This function creates an RSA key pair and saves the public and private keys as PEM files : `public_key.pem`, `public_key.pem`. The public key is used to protect the encryption session key, while the private key is needed to recover it during decryption.

In [ ]:
from pathlib import Path

from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa


def generate_key_pair(output_dir="keys"):
    """Generate and save an RSA public/private key pair."""

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )

    public_key = private_key.public_key()

    private_path = output_dir / "private_key.pem"
    public_path = output_dir / "public_key.pem"

    private_path.write_bytes(
        private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        )
    )

    public_path.write_bytes(
        public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        )
    )

    print(f"Public key saved to: {public_path}")
    print(f"Private key saved to: {private_path}")

    return str(public_path), str(private_path)

if __name__ == "__main__":
    generate_key_pair()

Public key saved to: keys/public_key.pem
Private key saved to: keys/private_key.pem


### c) Define Cryptographic Helpers and Key-Handling Functions
This cell defines shared helpers for Base64 conversion, loading RSA keys, deriving separate AES keys with HKDF, and creating authenticated metadata. These utilities support consistent, authenticated encryption and decryption of individual PDF elements.

In [ ]:

import base64
import json
import os
from pathlib import Path

import fitz
import pandas as pd

from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


PAYLOAD_NAME = "semantic_encryption_payload.json"

SENSITIVE_LEVELS = {"Medium", "High"}


def b64e(data: bytes) -> str:
    return base64.b64encode(data).decode("ascii")


def b64d(value: str) -> bytes:
    return base64.b64decode(value, validate=True)


def derive_word_key(master_key: bytes, record_id: str) -> bytes:
    """Derive a separate AES-256 key for each encrypted word."""
    return HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=("pdf-word-encryption-v2:" + record_id).encode("utf-8"),
    ).derive(master_key)


def make_aad(
    record_id: str,
    page_number: int,
    word_index: int,
    sensitivity: str,
) -> bytes:
    """Authenticated metadata: changing it makes decryption fail."""
    return json.dumps(
        {
            "record_id": record_id,
            "page": page_number,
            "word_index": word_index,
            "sensitivity": sensitivity,
            "format_version": 2,
        },
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")


def load_public_key(path: str | Path):
    with open(path, "rb") as f:
        key = serialization.load_pem_public_key(f.read())

    return key


def load_private_key(path: str | Path):
    with open(path, "rb") as f:
        key = serialization.load_pem_private_key(
            f.read(),
            password=None,
        )

    return key


def load_sidecars(
    layout_path: str | Path,
    tokens_path: str | Path,
):
    with open(layout_path, "r", encoding="utf-8") as f:
        layout = json.load(f)

    # Handles UTF-8 BOM if the CSV was saved by Excel.
    tokens = pd.read_csv(tokens_path, encoding="utf-8-sig")

    required = {"page", "word_index", "text", "sensitivity"}
    missing = required - set(tokens.columns)

    if missing:
        raise ValueError(
            f"tokens_with_sensitivity.csv is missing columns: "
            f"{sorted(missing)}"
        )

    tokens = tokens.copy()
    tokens["page"] = pd.to_numeric(tokens["page"], errors="raise").astype(int)
    tokens["word_index"] = pd.to_numeric(
        tokens["word_index"], errors="raise"
    ).astype(int)
    tokens["sensitivity"] = tokens["sensitivity"].astype(str).str.strip()

    allowed = {"Low", "Medium", "High"}
    invalid = set(tokens["sensitivity"].dropna()) - allowed

    if invalid:
        raise ValueError(f"Unexpected sensitivity labels: {sorted(invalid)}")

    if tokens.duplicated(["page", "word_index"]).any():
        raise ValueError(
            "Duplicate (page, word_index) keys in the sensitivity CSV."
        )

    layout_words = {}

    for page_record in layout.get("pages", []):
        page_number = int(page_record["page"])

        for word in page_record.get("words", []):
            key = (page_number, int(word["word_index"]))

            if key in layout_words:
                raise ValueError(f"Duplicate word key in layout: {key}")

            layout_words[key] = word

    csv_keys = set(
        zip(tokens["page"].tolist(), tokens["word_index"].tolist())
    )
    layout_keys = set(layout_words)

    if csv_keys != layout_keys:
        missing_from_layout = sorted(csv_keys - layout_keys)[:10]
        missing_from_csv = sorted(layout_keys - csv_keys)[:10]

        raise ValueError(
            "CSV/layout word keys do not match. "
            f"Missing from layout (first 10): {missing_from_layout}; "
            f"missing from CSV (first 10): {missing_from_csv}"
        )

    # Check that the sidecar's word text agrees with the layout.
    for row in tokens.itertuples(index=False):
        key = (int(row.page), int(row.word_index))
        layout_text = str(layout_words[key]["text"])
        csv_text = str(row.text)

        if layout_text != csv_text:
            raise ValueError(
                f"Text mismatch at {key}: "
                f"layout={layout_text!r}, CSV={csv_text!r}"
            )

    return layout, tokens, layout_words


def choose_font_alias(word: dict) -> str:
    """Use a built-in PDF font approximation when restoring text."""
    bold = word.get("bold") is True
    italic = word.get("italic") is True

    if bold and italic:
        return "hebi"
    if bold:
        return "hebo"
    if italic:
        return "heit"

    return "helv"


def normalized_rgb(rgb):
    if not isinstance(rgb, list) or len(rgb) != 3:
        return (0, 0, 0)

    return tuple(
        max(0.0, min(1.0, float(channel) / 255.0))
        for channel in rgb
    )


def word_insert_point(word: dict) -> fitz.Point:
    """
    Estimate a text baseline from the word bbox.

    The extractor's span origin is not necessarily the origin of
    each individual word, so we do not reuse it for every word.
    """
    x0, y0, x1, y1 = map(float, word["bbox"])
    height = max(1.0, y1 - y0)

    return fitz.Point(x0, y1 - max(1.0, height * 0.18))


### d) Implement Selective PDF Encryption and Decryption
This module contains the main PDF-protection logic. It uses token sensitivity labels to determine which content to encrypt, applies authenticated AES-GCM encryption, and uses RSA-OAEP to protect the session key. It also prepares a sanitized PDF and the encrypted payload needed to reconstruct protected content during decryption.

In [ ]:
"""
pdf_protection.py

Selective PDF protection:
- Low sensitivity    -> AES-128
- Medium sensitivity -> AES-192
- High sensitivity   -> AES-256
- A random session key is wrapped using RSA-OAEP-SHA256.
- HKDF derives independent AES keys for each sensitivity tier.
- AES-GCM encrypts every extracted word and every original image patch.
- The encrypted PDF contains sanitized raster pages and encrypted payload.
- The session key is stored separately in encrypted_session_key.json.

Expected layout JSON:
{
    "input_pdf": {"sha256": "..."},
    "pages": [
        {
            "page": 1,
            "width_points": 612,
            "height_points": 792,
            "words": [
                {
                    "word_index": 0,
                    "text": "Example",
                    "bbox": [10, 20, 60, 35],
                    "source": "native"
                }
            ]
        }
    ]
}

Expected token CSV columns:
page, word_index, text, sensitivity

"sensitivity_level" is also accepted instead of "sensitivity".
"""

import base64
import hashlib
import io
import json
import os
import sys
from pathlib import Path

import fitz
import pandas as pd
from PIL import Image, ImageDraw
from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

TIERS = {
    "Low": 16,       # AES-128
    "Medium": 24,    # AES-192
    "High": 32,      # AES-256
}

TIER_RANK = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
}

PAYLOAD_NAME = "semantic_encryption_payload_v3.json"
SESSION_KEY_NAME = "encrypted_session_key.json"
PAYLOAD_VERSION = 3


# ---------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------

def b64e(data: bytes) -> str:
    return base64.b64encode(data).decode("ascii")


def b64d(value: str) -> bytes:
    return base64.b64decode(value.encode("ascii"))


def canonical_json(data) -> bytes:
    """Stable JSON encoding for AES-GCM additional authenticated data."""
    return json.dumps(
        data,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")


def sha256_file(path: str) -> str:
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_sensitivity(value) -> str:
    if value is None or pd.isna(value):
        raise ValueError("A token has a missing sensitivity label.")

    value = str(value).strip().capitalize()

    if value not in TIERS:
        raise ValueError(
            f"Invalid sensitivity {value!r}. "
            f"Allowed values: {', '.join(TIERS)}."
        )

    return value


def normalize_bbox(bbox):
    if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
        raise ValueError(f"Invalid bounding box: {bbox!r}")

    x0, y0, x1, y1 = map(float, bbox)

    if x1 < x0:
        x0, x1 = x1, x0

    if y1 < y0:
        y0, y1 = y1, y0

    return [x0, y0, x1, y1]


def clamp_bbox(bbox, width, height):
    x0, y0, x1, y1 = bbox

    x0 = max(0, min(int(round(x0)), width))
    y0 = max(0, min(int(round(y0)), height))
    x1 = max(0, min(int(round(x1)), width))
    y1 = max(0, min(int(round(y1)), height))

    if x1 <= x0:
        x1 = min(width, x0 + 1)

    if y1 <= y0:
        y1 = min(height, y0 + 1)

    if x1 <= x0 or y1 <= y0:
        raise ValueError(f"Bounding box is outside the page: {bbox}")

    return [x0, y0, x1, y1]


# ---------------------------------------------------------------------
# RSA key loading and session-key protection
# ---------------------------------------------------------------------

def load_public_key(path: str):
    with open(path, "rb") as file:
        key = serialization.load_pem_public_key(file.read())

    if not isinstance(key, rsa.RSAPublicKey):
        raise TypeError("The public key must be an RSA public key.")

    return key


def load_private_key(path: str, password=None):
    with open(path, "rb") as file:
        key = serialization.load_pem_private_key(
            file.read(),
            password=password,
        )

    if not isinstance(key, rsa.RSAPrivateKey):
        raise TypeError("The private key must be an RSA private key.")

    return key


def wrap_session_key(session_key: bytes, public_key) -> bytes:
    """Encrypt the random session key using the recipient's RSA public key."""
    return public_key.encrypt(
        session_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )


def unwrap_session_key(wrapped_key: bytes, private_key) -> bytes:
    """Recover the session key using the recipient's RSA private key."""
    return private_key.decrypt(
        wrapped_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )


def derive_tier_keys(session_key: bytes, salt: bytes) -> dict:
    """Derive independent AES keys for Low, Medium and High."""
    result = {}

    for tier, key_length in TIERS.items():
        result[tier] = HKDF(
            algorithm=hashes.SHA256(),
            length=key_length,
            salt=salt,
            info=f"pdf-protection:v{PAYLOAD_VERSION}:tier:{tier}".encode(),
        ).derive(session_key)

    return result


# ---------------------------------------------------------------------
# Token CSV and layout validation
# ---------------------------------------------------------------------

def load_layout(layout_path: str) -> dict:
    with open(layout_path, "r", encoding="utf-8") as file:
        layout = json.load(file)

    if "pages" not in layout or not isinstance(layout["pages"], list):
        raise ValueError("Layout JSON must contain a 'pages' list.")

    return layout


def get_page_number_column(df):
    for column in ("page", "page_number"):
        if column in df.columns:
            return column

    raise ValueError("Token CSV needs a 'page' or 'page_number' column.")


def build_sensitivity_map(tokens_path: str) -> dict:
    """
    Create a mapping keyed by (page_number, word_index).

    Duplicate keys are rejected rather than silently overwriting labels.
    """
    df = pd.read_csv(tokens_path)

    page_column = get_page_number_column(df)

    if "word_index" not in df.columns:
        raise ValueError("Token CSV needs a 'word_index' column.")

    sensitivity_column = None

    for candidate in ("sensitivity", "sensitivity_level"):
        if candidate in df.columns:
            sensitivity_column = candidate
            break

    if sensitivity_column is None:
        raise ValueError(
            "Token CSV needs a 'sensitivity' or 'sensitivity_level' column."
        )

    result = {}

    for _, row in df.iterrows():
        if pd.isna(row[page_column]) or pd.isna(row["word_index"]):
            raise ValueError("Token CSV contains a missing page or word_index.")

        page_number = int(row[page_column])
        word_index = int(row["word_index"])
        key = (page_number, word_index)

        if key in result:
            raise ValueError(
                f"Duplicate token key {key}. "
                "Each (page, word_index) must be unique."
            )

        result[key] = normalize_sensitivity(row[sensitivity_column])

    if not result:
        raise ValueError("Token CSV contains no tokens.")

    return result


def validate_layout_against_pdf(layout: dict, pdf: fitz.Document):
    if len(layout["pages"]) != len(pdf):
        raise ValueError(
            f"Page-count mismatch: layout has {len(layout['pages'])} pages "
            f"but PDF has {len(pdf)} pages."
        )

    page_numbers = [int(item["page"]) for item in layout["pages"]]

    if len(set(page_numbers)) != len(page_numbers):
        raise ValueError("Layout contains duplicate page numbers.")

    # The implementation uses the page values in the layout/CSV as labels.
    # For accessing the PDF, page 1 maps to PDF index 0.
    # If your layout is zero-based, use zero-based page labels consistently
    # in both layout JSON and token CSV, and adjust pdf_index_for_page().
    return True


def pdf_index_for_page(page_number: int, layout: dict) -> int:
    page_numbers = [int(item["page"]) for item in layout["pages"]]

    if 0 in page_numbers:
        index = page_number
    else:
        index = page_number - 1

    if index < 0 or index >= len(layout["pages"]):
        raise ValueError(f"Invalid page number: {page_number}")

    return index


def get_layout_page(layout: dict, page_number: int) -> dict:
    for page in layout["pages"]:
        if int(page["page"]) == int(page_number):
            return page

    raise KeyError(f"Page {page_number} not found in layout.")


# ---------------------------------------------------------------------
# Word coordinates and rasterization
# ---------------------------------------------------------------------

def word_bbox_to_display_points(word: dict, pdf_page: fitz.Page) -> list:
    """
    Convert a word's PDF point bbox into displayed-page coordinates.

    Native text bboxes are assumed to use PyMuPDF's unrotated page
    coordinate system. OCR bboxes are assumed to already use displayed
    page coordinates. Confirm these assumptions for your extractor.
    """
    bbox = normalize_bbox(word["bbox"])

    source = str(word.get("source", "native")).lower()

    if source != "ocr" and pdf_page.rotation:
        rect = fitz.Rect(bbox) * pdf_page.rotation_matrix
        bbox = [rect.x0, rect.y0, rect.x1, rect.y1]

    return bbox


def points_bbox_to_pixels(
    bbox_points,
    scale: float,
    page_rect: fitz.Rect,
    pixel_width: int,
    pixel_height: int,
    padding_pixels: int = 2,
):
    x0, y0, x1, y1 = bbox_points

    # Display-page point coordinates are relative to page_rect's origin.
    x0 = (x0 - page_rect.x0) * scale
    y0 = (y0 - page_rect.y0) * scale
    x1 = (x1 - page_rect.x0) * scale
    y1 = (y1 - page_rect.y0) * scale

    return clamp_bbox(
        [
            x0 - padding_pixels,
            y0 - padding_pixels,
            x1 + padding_pixels,
            y1 + padding_pixels,
        ],
        pixel_width,
        pixel_height,
    )


def render_page(pdf_page: fitz.Page, dpi: int):
    scale = dpi / 72.0
    matrix = fitz.Matrix(scale, scale)

    pixmap = pdf_page.get_pixmap(
        matrix=matrix,
        alpha=False,
        colorspace=fitz.csRGB,
    )

    image = Image.frombytes(
        "RGB",
        (pixmap.width, pixmap.height),
        pixmap.samples,
    )

    return image, scale


def estimate_background(image: Image.Image, bbox, ring: int = 4):
    """
    Estimate the background from pixels around the word rectangle.
    This avoids always painting white boxes over colored backgrounds.
    """
    x0, y0, x1, y1 = bbox
    width, height = image.size

    outer = [
        max(0, x0 - ring),
        max(0, y0 - ring),
        min(width, x1 + ring),
        min(height, y1 + ring),
    ]

    crop = image.crop(tuple(outer)).convert("RGB")
    pixels = list(crop.getdata())

    if not pixels:
        return (255, 255, 255)

    # Use per-channel medians.
    channels = list(zip(*pixels))
    median_rgb = tuple(
        sorted(channel)[len(channel) // 2]
        for channel in channels
    )

    return median_rgb


# ---------------------------------------------------------------------
# Image patch grouping
# ---------------------------------------------------------------------

def rectangles_overlap(a, b):
    return not (
        a[2] < b[0]
        or b[2] < a[0]
        or a[3] < b[1]
        or b[3] < a[1]
    )


def union_bbox(a, b):
    return [
        min(a[0], b[0]),
        min(a[1], b[1]),
        max(a[2], b[2]),
        max(a[3], b[3]),
    ]


def merge_patch_components(word_items):
    """
    Merge overlapping word rectangles into patch components.

    A merged patch is encrypted using the highest sensitivity among
    all words intersecting that component. This prevents a lower-tier
    patch key from exposing neighboring higher-tier content.
    """
    components = []

    for item in word_items:
        rect = item["pixel_bbox"]
        merged = {
            "bbox": rect[:],
            "items": [item],
        }

        changed = True

        while changed:
            changed = False
            remaining = []

            for existing in components:
                if rectangles_overlap(merged["bbox"], existing["bbox"]):
                    merged["bbox"] = union_bbox(
                        merged["bbox"],
                        existing["bbox"],
                    )
                    merged["items"].extend(existing["items"])
                    changed = True
                else:
                    remaining.append(existing)

            components = remaining

        components.append(merged)

    # A final merge pass handles transitive overlaps.
    changed = True

    while changed:
        changed = False
        result = []

        while components:
            current = components.pop(0)
            index = 0

            while index < len(components):
                other = components[index]

                if rectangles_overlap(current["bbox"], other["bbox"]):
                    current["bbox"] = union_bbox(
                        current["bbox"],
                        other["bbox"],
                    )
                    current["items"].extend(other["items"])
                    components.pop(index)
                    changed = True
                else:
                    index += 1

            result.append(current)

        components = result

    for component in components:
        component["sensitivity"] = max(
            (item["sensitivity"] for item in component["items"]),
            key=lambda tier: TIER_RANK[tier],
        )

    return components


# ---------------------------------------------------------------------
# AES-GCM word and patch encryption
# ---------------------------------------------------------------------

def encrypt_bytes(data: bytes, key: bytes, aad: bytes) -> dict:
    nonce = os.urandom(12)
    ciphertext = AESGCM(key).encrypt(nonce, data, aad)

    return {
        "nonce": b64e(nonce),
        "ciphertext": b64e(ciphertext),
    }


def decrypt_bytes(record: dict, key: bytes, aad: bytes) -> bytes:
    nonce = b64d(record["nonce"])
    ciphertext = b64d(record["ciphertext"])

    return AESGCM(key).decrypt(nonce, ciphertext, aad)


def encrypt_word_text(
    text_value: str,
    page_number: int,
    word_index: int,
    sensitivity: str,
    tier_keys: dict,
):
    aad_data = {
        "type": "word",
        "page": page_number,
        "word_index": word_index,
        "sensitivity": sensitivity,
        "version": PAYLOAD_VERSION,
    }

    encrypted = encrypt_bytes(
        str(text_value).encode("utf-8"),
        tier_keys[sensitivity],
        canonical_json(aad_data),
    )

    encrypted.update({
        "page": page_number,
        "word_index": word_index,
        "sensitivity": sensitivity,
        "aad": aad_data,
    })

    return encrypted


def encrypt_patch(
    image: Image.Image,
    component: dict,
    page_number: int,
    patch_index: int,
    tier_keys: dict,
):
    bbox = component["bbox"]
    patch_image = image.crop(tuple(bbox))

    buffer = io.BytesIO()
    patch_image.save(buffer, format="PNG")
    patch_bytes = buffer.getvalue()

    tier = component["sensitivity"]

    aad_data = {
        "type": "image_patch",
        "page": page_number,
        "patch_index": patch_index,
        "bbox": bbox,
        "sensitivity": tier,
        "version": PAYLOAD_VERSION,
    }

    encrypted = encrypt_bytes(
        patch_bytes,
        tier_keys[tier],
        canonical_json(aad_data),
    )

    encrypted.update({
        "page": page_number,
        "patch_index": patch_index,
        "bbox": bbox,
        "sensitivity": tier,
        "aad": aad_data,
    })

    return encrypted


# ---------------------------------------------------------------------
# Payload attachment helpers
# ---------------------------------------------------------------------

def attach_payload(pdf_document: fitz.Document, payload: dict):
    pdf_document.embfile_add(
        PAYLOAD_NAME,
        json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":"),
        ).encode("utf-8"),
        filename=PAYLOAD_NAME,
        desc="Encrypted semantic PDF payload",
    )


def read_payload(pdf_document: fitz.Document) -> dict:
    names = pdf_document.embfile_names()

    if PAYLOAD_NAME not in names:
        raise ValueError(
            f"Encrypted PDF does not contain {PAYLOAD_NAME!r}."
        )

    raw = pdf_document.embfile_get(PAYLOAD_NAME)
    payload = json.loads(raw.decode("utf-8"))

    if payload.get("version") != PAYLOAD_VERSION:
        raise ValueError(
            f"Unsupported payload version: {payload.get('version')!r}"
        )

    return payload


# ---------------------------------------------------------------------
# Encryption workflow
# ---------------------------------------------------------------------

def encrypt_pdf(
    input_pdf_path: str,
    layout_path: str,
    tokens_path: str,
    public_key_path: str,
    output_dir: str,
    dpi: int = 300,
):
    input_pdf_path = str(Path(input_pdf_path).resolve())
    layout_path = str(Path(layout_path).resolve())
    tokens_path = str(Path(tokens_path).resolve())
    public_key_path = str(Path(public_key_path).resolve())

    output_dir = Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    encrypted_pdf_path = output_dir / "encrypted_document.pdf"
    session_file_path = output_dir / SESSION_KEY_NAME

    if dpi < 72 or dpi > 1200:
        raise ValueError("DPI must be between 72 and 1200.")

    layout = load_layout(layout_path)
    sensitivity_map = build_sensitivity_map(tokens_path)
    public_key = load_public_key(public_key_path)

    input_hash = sha256_file(input_pdf_path)
    expected_hash = (
        layout.get("input_pdf", {}).get("sha256")
        if isinstance(layout.get("input_pdf"), dict)
        else None
    )

    if expected_hash and expected_hash.lower() != input_hash.lower():
        raise ValueError(
            "The input PDF SHA-256 does not match the hash recorded in "
            "the layout JSON. Ensure the layout was generated from this PDF."
        )

    original_pdf = fitz.open(input_pdf_path)
    validate_layout_against_pdf(layout, original_pdf)

    # One random 256-bit session key per encryption operation.
    session_key = AESGCM.generate_key(bit_length=256)

    # Salt is not secret; it is needed to reproduce the derived keys.
    hkdf_salt = os.urandom(16)
    tier_keys = derive_tier_keys(session_key, hkdf_salt)

    wrapped_key = wrap_session_key(session_key, public_key)

    session_document = {
        "version": PAYLOAD_VERSION,
        "algorithm": "RSA-OAEP-SHA256",
        "kdf": "HKDF-SHA256",
        "salt": b64e(hkdf_salt),
        "wrapped_session_key": b64e(wrapped_key),
        "rsa_key_size": public_key.key_size,
        "source_pdf_sha256": input_hash,
    }

    # Store only the RSA-wrapped session key, never the plaintext key.
    with open(session_file_path, "w", encoding="utf-8") as file:
        json.dump(session_document, file, indent=2)

    payload = {
        "version": PAYLOAD_VERSION,
        "description": "Encrypted semantic PDF payload",
        "dpi": dpi,
        "source_pdf_sha256": input_hash,
        "pages": [],
    }

    sanitized_pdf = fitz.open()

    try:
        for layout_page in layout["pages"]:
            page_number = int(layout_page["page"])
            pdf_index = pdf_index_for_page(page_number, layout)
            source_page = original_pdf[pdf_index]

            original_image, scale = render_page(source_page, dpi)
            pixel_width, pixel_height = original_image.size
            page_rect = source_page.rect

            page_words = layout_page.get("words", [])
            word_items = []
            encrypted_words = []

            # Match every layout word to its sensitivity label.
            for word in page_words:
                if "word_index" not in word or "text" not in word:
                    raise ValueError(
                        f"Page {page_number} has a word without "
                        "'word_index' or 'text'."
                    )

                word_index = int(word["word_index"])
                key = (page_number, word_index)

                if key not in sensitivity_map:
                    raise ValueError(
                        f"No sensitivity label found for token {key}."
                    )

                sensitivity = sensitivity_map[key]
                word_text = str(word["text"])

                # Encrypt the extracted word text for every tier.
                encrypted_words.append(
                    encrypt_word_text(
                        word_text,
                        page_number,
                        word_index,
                        sensitivity,
                        tier_keys,
                    )
                )

                display_bbox = word_bbox_to_display_points(
                    word,
                    source_page,
                )

                pixel_bbox = points_bbox_to_pixels(
                    display_bbox,
                    scale,
                    page_rect,
                    pixel_width,
                    pixel_height,
                    padding_pixels=2,
                )

                word_items.append({
                    "page": page_number,
                    "word_index": word_index,
                    "text": word_text,
                    "sensitivity": sensitivity,
                    "pixel_bbox": pixel_bbox,
                })

            # Encrypt original pixel patches. Overlapping patches are
            # combined and protected at the strongest sensitivity tier.
            components = merge_patch_components(word_items)
            encrypted_patches = []

            for patch_index, component in enumerate(components):
                encrypted_patches.append(
                    encrypt_patch(
                        original_image,
                        component,
                        page_number,
                        patch_index,
                        tier_keys,
                    )
                )

            # Make a sanitized copy by masking the extracted word regions.
            sanitized_image = original_image.copy()
            draw = ImageDraw.Draw(sanitized_image)

            for item in word_items:
                bbox = item["pixel_bbox"]
                background = estimate_background(
                    original_image,
                    bbox,
                )

                draw.rectangle(
                    tuple(bbox),
                    fill=background,
                )

            # The encrypted PDF contains only the sanitized raster page,
            # not the original PDF text or original embedded images.
            image_buffer = io.BytesIO()
            sanitized_image.save(image_buffer, format="PNG")

            displayed_rect = source_page.rect
            sanitized_page = sanitized_pdf.new_page(
                width=displayed_rect.width,
                height=displayed_rect.height,
            )

            sanitized_page.insert_image(
                sanitized_page.rect,
                stream=image_buffer.getvalue(),
            )

            payload["pages"].append({
                "page": page_number,
                "page_index": pdf_index,
                "width_points": displayed_rect.width,
                "height_points": displayed_rect.height,
                "pixel_width": pixel_width,
                "pixel_height": pixel_height,
                "words": encrypted_words,
                "patches": encrypted_patches,
            })

            print(
                f"Encrypted page {page_number}: "
                f"{len(encrypted_words)} words, "
                f"{len(encrypted_patches)} image patches."
            )

        attach_payload(sanitized_pdf, payload)

        sanitized_pdf.save(
            str(encrypted_pdf_path),
            garbage=4,
            deflate=True,
        )

    finally:
        sanitized_pdf.close()
        original_pdf.close()

    print("\nEncryption completed.")
    print(f"Encrypted PDF: {encrypted_pdf_path}")
    print(f"Encrypted session key: {session_file_path}")


# ---------------------------------------------------------------------
# Decryption workflow
# ---------------------------------------------------------------------

def decrypt_pdf(
    encrypted_pdf_path: str,
    session_key_path: str,
    private_key_path: str,
    output_pdf_path: str,
):
    encrypted_pdf_path = str(Path(encrypted_pdf_path).resolve())
    session_key_path = str(Path(session_key_path).resolve())
    private_key_path = str(Path(private_key_path).resolve())
    output_pdf_path = str(Path(output_pdf_path).resolve())

    output_parent = Path(output_pdf_path).parent
    output_parent.mkdir(parents=True, exist_ok=True)

    with open(session_key_path, "r", encoding="utf-8") as file:
        session_document = json.load(file)

    if session_document.get("version") != PAYLOAD_VERSION:
        raise ValueError(
            f"Unsupported session-key file version: "
            f"{session_document.get('version')!r}"
        )

    private_key = load_private_key(private_key_path)

    wrapped_key = b64d(session_document["wrapped_session_key"])
    session_key = unwrap_session_key(wrapped_key, private_key)
    hkdf_salt = b64d(session_document["salt"])

    if len(session_key) != 32:
        raise ValueError("Unwrapped session key is not 256 bits.")

    tier_keys = derive_tier_keys(session_key, hkdf_salt)

    encrypted_pdf = fitz.open(encrypted_pdf_path)

    try:
        payload = read_payload(encrypted_pdf)

        if len(payload["pages"]) != len(encrypted_pdf):
            raise ValueError(
                "Payload page count does not match encrypted PDF page count."
            )

        restored_pdf = fitz.open()

        try:
            for pdf_index, page_record in enumerate(payload["pages"]):
                sanitized_page = encrypted_pdf[pdf_index]
                dpi = int(payload["dpi"])
                scale = dpi / 72.0

                pixmap = sanitized_page.get_pixmap(
                    matrix=fitz.Matrix(scale, scale),
                    alpha=False,
                    colorspace=fitz.csRGB,
                )

                restored_image = Image.frombytes(
                    "RGB",
                    (pixmap.width, pixmap.height),
                    pixmap.samples,
                )

                # Decrypt and authenticate every patch, then restore it
                # to the exact pixel rectangle recorded during encryption.
                for patch_record in page_record["patches"]:
                    tier = normalize_sensitivity(
                        patch_record["sensitivity"]
                    )

                    aad = canonical_json(patch_record["aad"])

                    try:
                        patch_bytes = decrypt_bytes(
                            patch_record,
                            tier_keys[tier],
                            aad,
                        )
                    except InvalidTag as exc:
                        raise ValueError(
                            "Image patch authentication failed. The "
                            "encrypted PDF, session key file, or private "
                            "key may be incorrect or corrupted."
                        ) from exc

                    with Image.open(io.BytesIO(patch_bytes)) as patch_image:
                        patch_image = patch_image.convert("RGB")
                        bbox = patch_record["bbox"]
                        expected_size = (
                            int(bbox[2]) - int(bbox[0]),
                            int(bbox[3]) - int(bbox[1]),
                        )

                        if patch_image.size != expected_size:
                            raise ValueError(
                                f"Patch size mismatch on page "
                                f"{page_record['page']}."
                            )

                        restored_image.paste(
                            patch_image,
                            (int(bbox[0]), int(bbox[1])),
                        )

                # Authenticate/decrypt every text record as well.
                # The restored pixels are the visual reconstruction;
                # decrypted text is validated and available for future
                # searchable-text-layer reconstruction if desired.
                decrypted_word_texts = []

                for word_record in page_record["words"]:
                    tier = normalize_sensitivity(
                        word_record["sensitivity"]
                    )

                    aad = canonical_json(word_record["aad"])

                    try:
                        plaintext = decrypt_bytes(
                            word_record,
                            tier_keys[tier],
                            aad,
                        )
                    except InvalidTag as exc:
                        raise ValueError(
                            "Word authentication failed. The encrypted "
                            "PDF, session key file, or private key may be "
                            "incorrect or corrupted."
                        ) from exc

                    decrypted_word_texts.append({
                        "word_index": word_record["word_index"],
                        "text": plaintext.decode("utf-8"),
                        "sensitivity": tier,
                    })

                # Create a page with the restored raster image.
                page_width = float(page_record["width_points"])
                page_height = float(page_record["height_points"])

                restored_page = restored_pdf.new_page(
                    width=page_width,
                    height=page_height,
                )

                image_buffer = io.BytesIO()
                restored_image.save(image_buffer, format="PNG")

                restored_page.insert_image(
                    restored_page.rect,
                    stream=image_buffer.getvalue(),
                )

                print(
                    f"Decrypted page {page_record['page']}: "
                    f"{len(decrypted_word_texts)} words, "
                    f"{len(page_record['patches'])} patches restored."
                )

            restored_pdf.save(
                output_pdf_path,
                garbage=4,
                deflate=True,
            )

        finally:
            restored_pdf.close()

    finally:
        encrypted_pdf.close()

    print("\nDecryption completed.")
    print(f"Restored PDF: {output_pdf_path}")

### e) Create Convenient Encryption and Decryption Wrappers
These wrapper functions pass the input PDF, layout metadata, sensitivity-labelled tokens, key files, and output paths to the main encryption/decryption functions. They centralize the file-path configuration and print where the generated outputs are saved.

In [ ]:

def run_encryption(
    input_pdf,
    layout="/content/data/processed/pdf_layout.json",
    tokens="/content/data/processed/tokens_with_sensitivity.csv",
    public_key="/content/keys/public_key.pem",
    output_dir="/content/data/protected",
    dpi=300,
):
    """Encrypt a PDF using the configured input files."""

    encrypt_pdf(
        input_pdf_path=input_pdf,
        layout_path=layout,
        tokens_path=tokens,
        public_key_path=public_key,
        output_dir=output_dir,
        dpi=dpi,
    )

    print("\nEncryption output files:")
    print(f"PDF: {output_dir}/encrypted_document.pdf")
    print(f"Session key: {output_dir}/encrypted_session_key.json")


def run_decryption(
    encrypted_pdf="/content/data/protected/encrypted_document.pdf",
    session_key="/content/data/protected/encrypted_session_key.json",
    private_key="/content/keys/private_key.pem",
    output_pdf="/content/data/protected/decrypted_document.pdf",
):
    """Decrypt the protected PDF using the configured key files."""

    decrypt_pdf(
        encrypted_pdf_path=encrypted_pdf,
        session_key_path=session_key,
        private_key_path=private_key,
        output_pdf_path=output_pdf,
    )

    print(f"\nDecrypted PDF saved to: {output_pdf}")

### f) Run the PDF Encryption Workflow
This call starts encryption for the specified input PDF using the configured layout, token labels, and public key. It produces the protected PDF and the separately stored encrypted session-key file.

In [ ]:
run_encryption(input_pdf="/content/Medical_Data_Document.pdf")

Encrypted page 1: 202 words, 183 image patches.
Encrypted page 2: 269 words, 229 image patches.

Encryption completed.
Encrypted PDF: /content/data/protected/encrypted_document.pdf
Encrypted session key: /content/data/protected/encrypted_session_key.json

Encryption output files:
PDF: /content/data/protected/encrypted_document.pdf
Session key: /content/data/protected/encrypted_session_key.json


### g) Run the PDF Decryption Workflow
 This call decrypts the protected PDF using the encrypted session-key file and the matching private key. It writes a reconstructed PDF so the protection workflow can be checked end to end.

In [ ]:
run_decryption()

Decrypted page 1: 202 words, 183 patches restored.
Decrypted page 2: 269 words, 229 patches restored.

Decryption completed.
Restored PDF: /content/data/protected/decrypted_document.pdf

Decrypted PDF saved to: /content/data/protected/decrypted_document.pdf
